# Achtergrond & Biografie van Politici

**Thema:** Opleidingsachtergrond, loopbaan en demografie.

**Kernvragen:** Wat is de gemiddelde leeftijd, opleidingsachtergrond en geografische spreiding (geboorteplaats versus woonplaats) van politici? Welke werkgevers domineren hun eerdere loopbaan?

Dit notebook verkent o.a. de silver-tabellen `persoon`, `persoon_onderwijs` en `persoon_loopbaan`.

In [ ]:
from pathlib import Path

import duckdb
import pandas as pd
import matplotlib.pyplot as plt

def find_database(start: Path | None = None) -> Path:
    """Find the repository DuckDB database by walking up from the current folder."""
    current = (start or Path.cwd()).resolve()
    for folder in (current, *current.parents):
        candidate = folder / "tweedekamer.duckdb"
        if candidate.is_file():
            return candidate
    raise FileNotFoundError("Could not find tweedekamer.duckdb from the current directory")


database_path = find_database()
connection = duckdb.connect(str(database_path), read_only=True) #connectie met de huidige database


def query_df(sql: str, parameters=None) -> pd.DataFrame:     #functie waarmee je sql kan schrijven
    """Execute a SQL query and return the result as a pandas DataFrame."""
    return connection.execute(sql, parameters or []).fetchdf()


database_path

In [ ]:
persoondfdf= query_df("""select *
from silver.persoon""")
persoondfdf

## EDA
Met het dashboard worden de volgende vragen behandeld: Wat is de gemiddelde leeftijd, opleidingsachtergrond en geografische spreiding (geboorteplaats versus woonplaats) van politici? Welke werkgevers domineren hun eerdere loopbaan? En zijn deze vragen te categoriseren op fractie?


### Populatie

In [ ]:
import pandas as pd

#set met waarden die beschouwd worden als geen echte informatie 
PLACEHOLDERS = {"", "-", "onbekend", "n.v.t.", "nan", "none","geen opgave", "geen"}

#alles selecteren van persoonsgegevens
persoon_ruw = query_df("SELECT * FROM silver.persoon")


print(persoon_ruw["id"].is_unique) #true = geen dubbele ID's
print("ruw:", len(persoon_ruw)) #totaal aantal mensen
print(persoon_ruw["functie"].value_counts()) #telt hoevaak een waarde voorkomt

persoon = persoon_ruw
print("Rijen :", len(persoon))   


- De populatie is 3083 personen met een uniek id: 150 Tweede Kamerleden, 85 Eerste Kamerleden en 2848 oud-Kamerleden.
- Aanname: een eerste kamerlid is ook een tweede kamerlid geweest omdat deze personen in de tweede kamer database voorkomen. Dashboard: Eerste Kamerleden apart tonen.

In [ ]:
# isna() mist placeholders; ze moeten als missing behandeld worden (PLACEHOLDERS, functie naar_na())
def naar_na(df):
    d = df.copy()
    for c in d.columns:
        if not (pd.api.types.is_numeric_dtype(d[c]) or pd.api.types.is_datetime64_any_dtype(d[c])): #api.types is return to pandas value
            leeg = d[c].astype(str).str.strip().str.lower().isin(PLACEHOLDERS)
            d[c] = d[c].mask(leeg, pd.NA)
    return d

persoon["geboortejaar"] = pd.to_datetime(persoon["geboortedatum"], errors="coerce").dt.year
persoon["decennium"] = persoon["geboortejaar"] // 10 * 10  #//= floordivision dus delen en naar beneden afronden

p = naar_na(persoon)
p2= p[["decennium","geslacht","geboortedatum_ruw","geboorteplaats","voornamen","overlijdensdatum_ruw","woonplaats"]]   
dekking = (p2.drop(columns=["decennium"]).notna()
             .groupby(p2["decennium"]).mean().mul(100).round(0))
dekking.insert(0, "n", p2.groupby("decennium").size())
dekking 

Geboortejaren lopen van de jaren 1750 tot en met 2000. Het grootste decennium is 1940 (293). Drie personen zonder geboortedatum vallen buiten de tabel (3080 van 3083). Voornamen zijn 0% ingevuld in 1750 en 100% vanaf 1980: dekking hangt dus af van het geboortejaar.

In [ ]:
"""
#### Handmatige correctie
- Voor mevrouw  I. (Iem) al Biyati ontbrak de geboortedatum als
enige van het huidig zittende parlement. 
Daardoor is er ook verder gekeken naar of ze een loopbaan heeft aangegeven en opleidingen.
Dit staat ook niet in de database. Om de data voor het huidige parlement aan te vullen
is de geboortedatum en geboorteplaats aangevuld. Loopbaan en onderwijs zijn niet nagelopen 
omdat ze dan anders wordt behandeld dan de andere 149 kamerleden die niet individueel zijn 
nagelopen.
- Dit geval doet wel vermoeden dat de database achterloopt op de huidige starters in het
 parlement. Dit is een aanwijzing geen bewijs
- Deze code moet boven de kwaliteitsprofiel tabel code gezet worden zodat deze accurate cijfers geeft

"""


pid = "01be9ad7-0444-483a-9983-5f66f1402f6a"

# stap 0: hulpkolommen altijd (opnieuw) aanmaken, dan staat de volgorde niet meer in de weg
persoon["geboorte_d"]   = pd.to_datetime(persoon["geboortedatum"], errors="coerce")
persoon["overlijden_d"] = pd.to_datetime(persoon["overlijdensdatum"], errors="coerce")
persoon["geboortejaar"] = persoon["geboorte_d"].dt.year
persoon["geboorteplaats"] = persoon["geboorteplaats"].astype("object")

m = persoon["id"] == pid
assert m.sum() == 1, f"verwacht 1 rij, gevonden {m.sum()}"

# 1. wat staat er in de bron? (alleen kolommen die bestaan)
kol = ["functie", "geboortedatum", "geboorteplaats", "woonplaats", "opleiding_status",
       "opleiding_hoogste", "n_loopbaanrijen", "n_buiten_zeker"]
print("ontbrekende kolommen:", [c for c in kol if c not in persoon.columns])
print(persoon.loc[m, [c for c in kol if c in persoon.columns]].to_string())

# 2. correctie vastleggen (alleen invullen wat in de bron leeg is)
CORRECTIES = {
    pid: {"geboortedatum": "1998-07-12", "geboorteplaats": "Amsterdam",
          "bron": "parlement.com", "opgezocht": "2026-10-08"},
}
persoon["handmatig_aangevuld"] = persoon["id"].isin(CORRECTIES)
for k, c in CORRECTIES.items():
    m = persoon["id"] == k
    assert m.sum() == 1
    d = pd.to_datetime(c["geboortedatum"])
    if persoon.loc[m, "geboorte_d"].isna().all():
        persoon.loc[m, "geboorte_d"] = d
        persoon.loc[m, "geboortejaar"] = d.year
    if pd.isna(persoon.loc[m, "geboorteplaats"].iloc[0]):
        persoon.loc[m, "geboorteplaats"] = c["geboorteplaats"]
    if persoon.loc[m, "geboortedatum"].isna().all():
        persoon.loc[m, "geboortedatum"] = c["geboortedatum"]

print(persoon.loc[m, ["geboorte_d", "geboortejaar", "geboorteplaats", "handmatig_aangevuld"]].to_string())

In [ ]:
#kwaliteitsprofiel per tabel

def profiel(df, tabel): #dataframe + naam tabel
    n = len(df) #aantal rijen
    rijen = [] 
    for kol in df.columns:
        s = df[kol]
        nan = int(s.isna().sum()) #tellen aantal NAN
        plh = int((s.notna() & s.astype(str).str.strip().str.lower().isin(PLACEHOLDERS)).sum()) #waardes die nvt of - zijn niet nan
        rijen.append({"tabel": tabel, "kolom": kol, "n": n, "nan": nan,
                      "ingevuld_%": round(100 * (n - nan - plh) / n, 1), #totaal- nan- placeholders
                      "placeholder": plh, #aantal placeholders
                      "unieke": s.nunique()})
    return pd.DataFrame(rijen)

loopbaan  = query_df("SELECT * FROM silver.persoon_loopbaan")
onderwijs = query_df("SELECT * FROM silver.persoon_onderwijs")
loopbaan  = loopbaan[loopbaan["persoon_id"].isin(persoon["id"])].copy()
onderwijs = onderwijs[onderwijs["persoon_id"].isin(persoon["id"])].copy()

kwaliteit = pd.concat([profiel(persoon, "persoon"),  #pd.concat=samenvoegen van de tabellen gegeven door de functie
                       profiel(loopbaan, "persoon_loopbaan"),
                       profiel(onderwijs, "persoon_onderwijs")], ignore_index=True)
kwaliteit

Geslacht en functie zijn 100% ingevuld, geboortedatum 99,9% (eerst 3 missende waarden. Na inspectie op de huidige kamerleden is dit getal nu 2 geworden), geboorteplaats 61,2%, woonplaats 55,3% en geboorteland 21,3%. De loopbaan heeft 1987 rijen over 278 personen (werkgever 83,9% ingevuld, functie 96,7%). Het onderwijs heeft 979 rijen over 278 personen (instelling 79,6%). Het percentage is per rij, niet per persoon.

In [ ]:
# hoeveel personen hebben zowel geboorteplaats als woonplaats echt ingevuld, en hoe zit het bij de huidige 150?


p = naar_na(persoon)      # placeholders ("-") worden hierin als leeg behandeld
dekking_functie = (p.groupby("functie")[["geboorteplaats", "woonplaats", "geboorteland"]]
                    .agg(lambda s: round(100 * s.notna().mean(), 1)))
dekking_functie.insert(0, "n", p.groupby("functie").size())
dekking_functie

De huidige Kamer is bijna volledig ingevuld (geboorteplaats 100% en woonplaats 99,3%, geboorteland 98,0%), de oud-Kamerleden niet (59,0%, 52,0%, 16,1%).

### Fractie
fractie_zetel_persoon  →  fractie_zetel  →  fractie
 (persoon + periode)       (de zetel)       (de partij)\
 \
 Deze join moet gemaakt worden om te kijken welke persoon er in welke fractie zit. Interessant om te kijken waar je een personen uit een fractie vandaan komen of hoe oud ze zijn.\
 Omdat persoon_id voorkomt in fractiezetelpersoon hoeft er niet nog een join gemaakt te worden tussen persoon en fractiezetelpersoon

In [ ]:
fractie = query_df("""
SELECT fzp.persoon_id, f.id AS fractie_id, f.afkorting, f.naam_nl,
       fzp.van, fzp.tot_en_met,
       ROW_NUMBER() OVER (PARTITION BY fzp.persoon_id
                          ORDER BY fzp.van ASC,  fzp.tot_en_met ASC  NULLS LAST)  AS rn_eerste,
       ROW_NUMBER() OVER (PARTITION BY fzp.persoon_id
                          ORDER BY fzp.van DESC, fzp.tot_en_met DESC NULLS FIRST) AS rn_laatste,
       COUNT(DISTINCT f.id) OVER (PARTITION BY fzp.persoon_id) AS n_fracties
FROM silver.fractie_zetel_persoon fzp
JOIN silver.fractie_zetel fz ON fz.id = fzp.fractie_zetel_id
JOIN silver.fractie f        ON f.id = fz.fractie_id
""")

kol = ["persoon_id", "fractie_id", "afkorting", "naam_nl", "van", "tot_en_met"]

eerste = (fractie[fractie["rn_eerste"] == 1][kol + ["n_fracties"]] #persoon bij de eerste fractie was
          .rename(columns={"fractie_id": "fractie_eerste_id", "afkorting": "fractie_eerste_afk",
                           "naam_nl": "fractie_eerste_naam", "van": "fractie_eerste_van",
                           "tot_en_met": "fractie_eerste_tot"})
          .set_index("persoon_id"))

laatste = (fractie[fractie["rn_laatste"] == 1][kol] #selecteren welke persoon bij laatste fractie was
           .rename(columns={"fractie_id": "fractie_laatste_id", "afkorting": "fractie_laatste_afk",
                            "naam_nl": "fractie_laatste_naam", "van": "fractie_laatste_van",
                            "tot_en_met": "fractie_laatste_tot"})
           .set_index("persoon_id"))

nieuw = list(eerste.columns) + list(laatste.columns)
persoon = persoon.drop(columns=[c for c in nieuw + ["persoon_id", "heeft_fractie", "start_jaar_fractie",
                                                    "partijwissel", "fractie_laatste_open"]
                                if c in persoon.columns])

n_voor = len(persoon)
persoon = (persoon.merge(eerste,  left_on="id", right_index=True, how="left")
                  .merge(laatste, left_on="id", right_index=True, how="left"))
assert len(persoon) == n_voor, "join heeft rijen verdubbeld"

persoon["heeft_fractie"]       = persoon["fractie_eerste_id"].notna()
persoon["start_jaar_fractie"]  = pd.to_datetime(persoon["fractie_eerste_van"], errors="coerce").dt.year
persoon["partijwissel"]        = persoon["n_fracties"] > 1  #persoon bij meerdere fracties heeft gezeten
persoon["fractie_laatste_open"] = persoon["heeft_fractie"] & persoon["fractie_laatste_tot"].isna()

print(persoon["heeft_fractie"].value_counts())    
print(persoon["partijwissel"].sum(), "personen met meer dan één fractie")

- 803 personen (26,0%) hebben een fractiekoppeling, 2280 niet. 76 personen hebben meer dan één fractie
- Partij-visuals alleen voor de 803, met noemer; de rest heet "geen fractiekoppeling in de bron".

### Onderwijs
- Met REGEX (regular expressions) wordt er gezocht op trefwoorden in strings per record. Via deze strings kan er een categorie aan de record worden gegeven. Dus persoon A werkt als directeur bij een bank -> bedrijfsleven. 
- Belangrijk om te vermelden is dat deze methode een ruwe schatting geeft.
Een directeur bij de GGD moet ingedeeld worden op zorg maar komt helaas toch in bedrijfsleven terecht. Er is geprobeerd om deze fouten te voorkomen door middel van steekproeven. Dit garandeerd alleen geen honderd procent succes

In [ ]:
#opleiding categoriseren 
import re
import pandas as pd

PLACEHOLDERS |= {"geen opgave", "geen"}


def schoon(t):
    t = str(t).lower()
    t = re.sub(r"(?<=\w)\.(?=\w)", "", t)      # v.w.o. -> vwo, m.t.s. -> mts
    t = t.replace(".", "").replace('"', "")
    return re.sub(r"\s+", " ", t).strip()


def leeg(t):
    return pd.isna(t) or schoon(t) in PLACEHOLDERS

#met .unique alle unieke waardes gezocht en met ChatGPT laten indelen op categorie

NIET_VOLTOOID = r"niet voltooid|niet afgerond|onvoltooid|gestopt|afgebroken|\(\d+ jaar\)|propedeuse"

# Niveau op basis van de opleidingstekst. Volgorde telt: de eerste die matcht wint.
REGELS_TEKST = [
    ("cursus",
     r"cursus|certificaat|training|leergang|workshop|masterclass|"
     r"executive|postgraduate|postdoctoraal|postdoctoral|"
     r"\bnive\b|\bnima\b|coach|toezichthouder|mediation|"
     r"ondernemersvaardigheden|basis ?kwalificatie|register controller|"
     r"erasmusbeurs|vooropleiding|secretaresse|bestuurder|"
     r"management development|management trainee|bijscholing|"
     r"inservice|in-service|avondopleiding|avondcursus|"
     r"beroepsopleiding|kadertraining|"
     r"management course|leadership course|summer school|introductie|\bminor\b"),

    ("promotie",
     r"promotie|phd|gepromoveerd|doctorate|doctoraat"),

    ("hbo_expliciet",   # 'university of applied sciences' moet hbo zijn, niet wo
     r"applied sciences|hoger beroepsonderwijs|higher vocational"),

    ("wo",
     r"master|msc|\bllm\b|doctoraal|\bdrs\b|\bir\b|ingenieur|"
     r"bachelor|\bba\b|\bbsc\b|\bma\b|\bmba\b|\bmphil\b|"
     r"kandidaats|universitair|\bwo\b|universitaire|university|doctorandus"),

    ("hbo",
     r"\bhbo\b|hogeschool|pabo|\bhts\b|\bheao\b|\bhes\b|"
     r"lerarenopleiding|kunstacademie|hotelschool"),

    ("mbo",
     r"\bk?mbo\b|\bmts\b|meao|middelbaar beroepsonderwijs|"
     r"ambachtsschool|\bito\b|kleuterleidster|verpleegkundige|"
     r"spd[- ]bedrijfsadministratie|vakopleiding"),

    ("vo",
     r"mavo|havo|vwo|gymnasium|atheneum|\bhbs\b|mulo|lyceum|"
     r"vmbo|\blbo\b|\bulo\b|middelbare|vbo|leao|mms|"
     r"algemeen voortgezet onderwijs|"
     r"secondary school|secondary education|highschool|high school"),

    ("basisonderwijs",
     r"basisschool|lagere school|primary school|primary education"),
]

# Niveau op basis van de instelling (terugval). Volgorde telt.
INSTELLING_REGELS = [
    ("hbo",
     r"hogeschool|hogescholen|applied sciences|\bhbo\b|\bheao\b|\bhes\b|"
     r"fontys|saxion|windesheim|avans|inholland|\bhva\b|\bhu\b|hanze|"
     r"academie van bouwkunst|hotelschool|militaire academie"),

    ("wo",
     r"universiteit|universitair|university|\buva\b|\bvu\b|\btu\b|"
     r"erasmus|radboud|nyenrode|\bkub\b|\brug\b|"
     r"london school|school of economics|sorbonne|oxford|\beur\b"),

    ("mbo",
     r"\broc\b|\bmbo\b|koning willem|albeda"),

    ("vo",
     r"gymnasium|lyceum|\bmavo\b|\bhavo\b|\bvwo\b|\bvbo\b|\bleao\b|"
     r"scholengemeenschap|college\b|middelbare school|secondary school"),

    ("basisonderwijs",
     r"basisschool|primary school|lagere school|primary education"),

    ("cursus",
     r"ministerie|orde van advocaten|vereniging|stichting|"
     r"psychodynamica|schoevers|sioo"),
]


def niveau_instelling(instelling):
    if leeg(instelling):
        return None
    i = schoon(instelling)
    for naam, pat in INSTELLING_REGELS:
        if re.search(pat, i):
            return naam
    return None


def niveau(opl, instelling):
    """Geeft (categorie, bron, status). bron = waar het niveau vandaan komt."""
    status = "voltooid"
    if not leeg(opl) and re.search(NIET_VOLTOOID, schoon(opl)):
        status = "niet_voltooid"

    if not leeg(opl):
        t = schoon(opl)
        for naam, pat in REGELS_TEKST:
            if re.search(pat, t):
                return ("hbo" if naam == "hbo_expliciet" else naam), "tekst", status

    ni = niveau_instelling(instelling)          # terugval: soort instelling, niet de graad
    if ni is not None:
        return ni, "instelling", status

    return ("onbekend" if leeg(opl) else "niet_ingedeeld"), "geen", status


CANON = [
    (r"universiteit van amsterdam|\buva\b", "Universiteit van Amsterdam"),
    (r"vrije universiteit(?! brussel)|\bvu\b", "Vrije Universiteit Amsterdam"),
    (r"leiden",                             "Universiteit Leiden"),
    (r"erasmus",                            "Erasmus Universiteit Rotterdam"),
    (r"groningen",                          "Rijksuniversiteit Groningen"),
    (r"utrecht",                            "Universiteit Utrecht"),
    (r"radboud|nijmegen",                   "Radboud Universiteit"),
    (r"tilburg|brabant",                    "Tilburg University"),
    (r"maastricht",                         "Universiteit Maastricht"),
    (r"twente",                             "Universiteit Twente"),
    (r"wageningen",                         "Wageningen University"),
    (r"delft",                              "TU Delft"),
    (r"nyenrode",                           "Nyenrode Business Universiteit"),
    (r"open universiteit",                  "Open Universiteit"),
    (r"oxford",                             "University of Oxford"),
]


def instelling_schoon(x):
    if leeg(x):
        return pd.NA
    if niveau_instelling(x) != "wo":
        return str(x).strip()
    s = schoon(x)
    for pat, naam in CANON:
        if re.search(pat, s):
            return naam
    return str(x).strip()


# toepassen op elke opleidingsrij van de onderwijstabel
onderwijs = onderwijs.drop(columns=[c for c in ["categorie", "bron", "status", "rang", "instelling_schoon"]
                                    if c in onderwijs.columns])
onderwijs[["categorie", "bron", "status"]] = onderwijs.apply(
    lambda r: niveau(r["opleiding_nl"], r["instelling"]), axis=1, result_type="expand")
onderwijs["instelling_schoon"] = onderwijs["instelling"].map(instelling_schoon)


# hoogste voltooide niveau per persoon
RANG = {"basisonderwijs": 0, "vo": 1, "mbo": 2, "hbo": 3, "wo": 4, "promotie": 5}  #dictionary
onderwijs["rang"] = onderwijs["categorie"].map(RANG)
geldig = onderwijs[(onderwijs["status"] == "voltooid") & onderwijs["rang"].notna()].copy()
# bij gelijke rang wint 'tekst' boven 'instelling' (True sorteert laatst, tail(1) pakt die)
geldig["bron_is_tekst"] = geldig["bron"] == "tekst"
hoogste = (geldig.sort_values(["rang", "bron_is_tekst"]).groupby("persoon_id").tail(1)
           [["persoon_id", "categorie", "bron"]]
           .rename(columns={"categorie": "opleiding_hoogste", "bron": "opleiding_hoogste_bron"})
           .set_index("persoon_id"))
n_opl = onderwijs.groupby("persoon_id").size().rename("n_opleidingen")

persoon = persoon.drop(columns=[c for c in ["opleiding_hoogste", "opleiding_hoogste_bron",
                                            "n_opleidingen", "opleiding_status"] if c in persoon.columns])
n_voor = len(persoon)
persoon = (persoon.merge(hoogste, left_on="id", right_index=True, how="left")
                  .merge(n_opl,   left_on="id", right_index=True, how="left"))
assert len(persoon) == n_voor, "merge heeft personen verdubbeld"
persoon["n_opleidingen"] = persoon["n_opleidingen"].fillna(0).astype(int)

persoon["opleiding_status"] = "geen opleiding geregistreerd"
persoon.loc[persoon["n_opleidingen"] > 0, "opleiding_status"] = "geregistreerd, geen voltooid niveau af te leiden"
persoon.loc[persoon["opleiding_hoogste"].notna(), "opleiding_status"] = "niveau bepaald"

# wat is ingedeeld en wat niet (per opleidingsrij)
onderwijs["ingedeeld"] = ~onderwijs["categorie"].isin(["niet_ingedeeld", "onbekend"])
print(onderwijs["ingedeeld"].value_counts())
print(onderwijs["categorie"].value_counts())

# wat het dashboard gebruikt (per persoon)
print(persoon["opleiding_status"].value_counts())
print(persoon["opleiding_hoogste"].value_counts())

print(persoon["opleiding_hoogste_bron"].value_counts())


Van de 979 opleidingsrijen zijn er 849 (86,7%) ingedeeld op niveau; 130 (13,3%) zijn niet ingedeeld of onbekend. Het hoogste voltooide niveau is bepaald voor 265 van de 3083 personen (8,6%): wo 193, hbo 30, vo 19, promotie 15, mbo 8. Bij 110 van die 265 (41,5%) is het niveau geschat op basis van de instelling, bij 155 komt het uit de opleidingstekst. 13 personen hebben opleidingsrijen maar geen af te leiden voltooid niveau, en 2805 hebben geen opleidingsrij. "Geen opleiding geregistreerd" is niet hetzelfde als "geen opleiding gevolgd". Cursussen en niet voltooide opleidingen tellen niet mee voor het hoogste niveau.

In [ ]:
print(pd.crosstab(persoon["decennium"], persoon["opleiding_status"], normalize="index").round(2).to_string())
print(pd.crosstab(persoon["functie"], persoon["opleiding_status"]).to_string())

Opleidingsniveau is vrijwel alleen bekend voor personen geboren vanaf de jaren 1950 (10% in 1950, 29% in 1960, 44% in 1970, 33% in 1980, 32% in 1990). Per functie: Tweede Kamer 56 van 150, Eerste Kamer 2 van 85, oud-Kamerlid 207 van 2848.

In [ ]:
# 1. aantallen erbij, alleen vanaf 1940
t = pd.crosstab(persoon["decennium"], persoon["opleiding_status"])
t["n"] = t.sum(axis=1)
t["bepaald_%"] = (100 * t["niveau bepaald"] / t["n"]).round(0)
print(t.loc[1940:].to_string())

# 2. hangt opleiding samen met de fractiedata? (dezelfde bron-periode?)
print(pd.crosstab(persoon["heeft_fractie"], persoon["opleiding_status"]).to_string())

# 3. alleen de 150 Tweede Kamerleden: wanneer zijn ze begonnen, bij welke fractie?
tk = persoon[persoon["functie"] == "Tweede Kamerlid"].copy()
tk["start_bin"] = pd.cut(tk["start_jaar_fractie"], [1976, 2006, 2012, 2017, 2021, 2023, 2026])
print(pd.crosstab(tk["start_bin"], tk["opleiding_status"]).to_string())
print(pd.crosstab(tk["fractie_eerste_afk"], tk["opleiding_status"]).to_string())

Onder de 2280 zonder fractie heeft niemand een opleidingsrij. Bij de huidige Kamer daalt het aandeel met een niveau naarmate de start recenter is: 6 van 7 (start tot 2006), 3 van 3, 8 van 12, 12 van 29, 12 van 31 en 15 van 68 (22%) voor starters sinds 2024.

### Loopbaan
Ook hier is REGEX een prima ruwe inschatting maar niet foutloos. Een directeur van safe the children wordt ingedeeld op bedrijfsleven terwijl een NGO (non governmental organisation) eigenlijk maatschappelijk is. Daarom wordt er ingedeeld op "bedrijfsleven en overige organisaties "

In [ ]:
loopbaan["functie"]   = loopbaan["functie"].fillna("")
loopbaan["werkgever"] = loopbaan["werkgever"].fillna("")
loopbaan["tekst"] = (loopbaan["functie"] + " | " + loopbaan["werkgever"]).str.lower()

POLITIEK_CONTEXT = (r"fractie|tweede kamer|eerste kamer|kamerlid|partij|campagne|verkiezing|wetenschappelijk bureau|"
                    r"\b(?:vvd|pvda|cda|d66|sp|gl|groenlinks|pvv|christenunie|cu|sgp|volt|denk|bbb|fvd)\b")



REGELS_SECTOR = [
    (
        "kamerlidmaatschap",
        r"kamerlid|tweede kamer(?: der staten-generaal)?|eerste kamer(?: der staten-generaal)?|senaatslid",
    ),
    (
        "politiek_ondersteuning",
        r"medewerk(?:er|ster)?|campagne|(?<!staats)secretaris|voorlichter|fractiebureau|kamerfractie|"
        r"woordvoerder|assistent|politiek(?:e)? (?:adviseur|assistent)|beleidsadviseur|"
        r"co[oö]rdinerend beleidsadviseur|\bbeleid\b|policy officer|politiek(?:e)? co[oö]rdinat|speechschrijver|"
        r"verkiezingsprogramma|parlementaire (?:redactie|en departementale contacten)|"
        r"advies(?:eur|ering).*(?:minister|parlement|fractie|verkiez|coalitie|kabinets|politiek|binnenlands bestuur)|"
        r"wetenschappelijk bureau (?:vvd|pvda|cda|d66|sp|gl|groenlinks|pvv|cu)|"
        r"fondsenwerver.*\b(?:vvd|pvda|cda|d66|sp|gl|groenlinks|pvv|christenunie|cu)\b",
    ),
    (
        "politiek_bestuur",
        r"fractievoorzitter|lijsttrekker|parlementslid|informateur|formateur|europees parlement|europarlementari(?:ë|e)r|"
        r"comit[eé] van de regio|vaste commissie|\blid db\b|\blid ab\b|"
        r"\bminister\b|minister-president|staatssecretaris|wethouder|burgemeester|"
        r"gemeenteraad|provinciale staten|gedeputeerde|raadslid|statenlid|statenfractie|"
        r"stadsdeelraad|stadsdeelvoorzitter|burgerstatenlid|duo-statenlid|"
        r"steunfractielid|fractievertegenwoordiger|bestuurscommissielid|partij|"
        r"\b(?:vvd|pvda|cda|d66|sp|gl|groenlinks|pvv|christenunie|cu|sgp|volt|denk|bbb|"
        r"forum voor democratie|fvd|partij voor de dieren)\b|cdja|christen democratisch jongeren|jonge democraten|jovd",
    ),
    (
        "media_communicatie",
        r"journalist|verslaggever|redacteur|hoofdredacteur|columnist|presentat|programmamaker|"
        r"omroep|televisie|\bradio\b|\bavro\b|\bnos\b|\bncrv\b|persvoorlichter|"
        r"nieuwslezer|redactielid|radio(?:columnist|culumnist)|documentairemaker|documentaire|toneel(?:makerij)?|actrice|acteur|voice[ -]?over|\bdj\b|"
        r"producent|publicist|tekstschrijver|mediabedrijf|communicatie\s*(?:adviseur|advies|manager|co[oö]rdinat)|"
        r"hoofd (?:corporate )?communicatie|hoofd informatie.*communicatie|hoofd voorlichting|\bvoorlichting\b",
    ),
    (
        "juridisch",
        r"advocaat|rechter|jurist|notaris|officier van justitie|fiscalist|belastingadviseur|"
        r"(?:company|legal) lawyer|legal advisor|bewindvoerder|mediator|mediation|procesvertegenwoordiger|"
        r"griffier|raadsheer|zittingsvertegenwoordiger|rechtstoepassing|personenschade|vennootschapsbelasting|(?:aanbestedings|staats)recht|"
        r"wetgevings|wet mulder|habeas corpus|formeel recht",
    ),
    (
        "onderwijs_wetenschap",
        r"universiteit|university|hogeschool|docent|leraar|hoogleraar|professor|onderwijs|school|"
        r"onderzoek|\bcito\b|examen|vmbo|\bvwo\b|\bhavo\b|\bhbo\b|postdoctoral|postdoc|"
        r"promovend|promotietraject|proefschrift|archeoloog|\blector\b|\brector\b|cursusleider|"
        r"afdelingsleider|stagebegeleider|studentadviseur|taalles|propedeuse|opleiding|\bects\b",
    ),
    (
        "zorg",
        r"\barts\b|ziekenhuis|verpleeg|zorg|huisarts|ggz|verloskund|laborant|pati[eë]nt|oncologie|"
        r"medicatieveiligheid|jeugdhulp|gezinsvoogd|jeugdwelzijn|maatschappelijk werker|"
        r"re[- ]integratie|gedragswetenschapper|beschermd wonen|preventieakkoord|\bggd\b|gezondheidsdienst",
    ),
    (
        "overheid",
        r"ministerie|\bgemeente|provincie|rijks|overheid|waterschap|ambassadeur|europese commissie|"
        r"belastingdienst|\bvng\b|ambtenaar|handhaver|belastingtelefoon|toeslagen|behandelfunctionaris|rekenkamer(?:commissie)?|"
        r"diplomaat|diplomatie|gezant|zaakgelastig|permanente vertegenwoordiger|"
        r"vertegenwoordiger van nederland|democratie en bestuur|inwonerszaken|"
        r"sociale zaken en werkgelegenheid|consulent wet inschakeling werkzoekenden|afdelingshoofd.*milieu|stafbureau|grondgebied|c2000|veiligheidsregio|volksraadpleging|openbare orde|"
        r"kabinetsaanpak|begrotingsbeleid|klantdirectie|stedelijke vernieuwing|decentralisat|democratisering|staatsadviescommissie|verkeer en vervoer|concentratiecontrole|planeconoom",
    ),
    (
        "maatschappelijk",
        r"vakbond|\bfnv\b|\bcnv\b|stichting|vereniging|belangen|\bbond\b|politiebond|"
        r"jeugdraad|oxfam|rotary|abvakabo|partos|diaken|ouderling|kerk|geloof|theologi|"
        r"vrijwillig|mensenrechten|rechten van de mens|opbouwwerk|sociaal[- ]cultureel|sociaal raadsvrouw|jeugdcommissie|"
        r"oudercommissie|leerlingenraad|studentenplatform|stop kindermisbruik",
    ),
    (
        "veiligheid_defensie",
        r"vlieg|squadron|\bf-16\b|\bpolitie\b|politieagent|militair|defensie|marine|luchtmacht|"
        r"marechaussee|rechercheur|hoof?dinspecteur|commandant|peloton|sergeant|officier|"
        r"legerkorps|reserve[- ]?officier|panzer|vuurmond|geleide wapens|vluchtcommandant|"
        r"wing scheduler|\bt-38\b|rsuto|runway supervisor|inlichtingen(?:-| )en veiligheids|veiligheidsvraagstuk",
    ),
    (
        "bedrijfsleven en overige organisaties",
        r"\bb\.?v\.?\b|\bn\.?v\.?\b|\bv\.?o\.?f\b|directeur|director|manager|management|"
        r"consultant|bank|verzekering|eigenaar|ondernemer|oprichter|vice president|controller|account|"
        r"marketing|sales|public affairs|\bhead of\b|personeelszaken|tester|adviesbureau|\bey\b|"
        r"ernst & young|deloitte|\bkpn\b|\b(?:cfo|ceo|dga)\b|chief economist|business|commercieel|"
        r"financieel|fixed income|kredietcrisis|investment|economist|analist|analyst|\bhr\b|\bp&o\b|"
        r"personeel|\bit(?:-er)?\b|\bict\b|systeem(?:beheer|ontwerp)|software|sysadmin|engineer|"
        r"constructeur|architect|bouw(?:techniek|kund|fysica)|vastgoed|technisch|kwaliteit|arbo|inkoop|"
        r"logistiek|bedrijfs(?:leider|voering|bureau)|industrie|retail|horeca|landbouw|melkvee|"
        r"boulanger|patissier|serveerster|chauffeur|machinist|verkoper|acquisiteur|reisbegeleid|"
        r"dier(?:enarts|enartsen)|consult|zzp|zelfstandig|freelance|\bowner\b|\bpartner\b|"
        r"treasurer|penningmeester|\btrainee\b|programmeur|projectontwikkelaar|procesontwikkeling|"
        r"ontwerper|teamleider|co[oö]rdinator|projectleider|programmaco[oö]rdinator|programmaleider|"
        r"directielid|directievoorzitter|\braad van bestuur\b|\braad van commissarissen\b|"
        r"\braad van toezicht\b|\bondernemingsraad\b|\badviseur\b|organisatie[- ]?adviseur|projectadviseur|bedrijvenadviseur|huisvestingsadviseur|subsidieadviseur|bedrijfsleid(?:er|ster)|leidinggevende|\bhead\b|senior associate|\btrainer\b|\bcoach\b|secretaresse|archivaresse|systeem[-/ ]*(?:netwerk)?beheer(?:der)?|quality inspector|\beigenares\b|\becono(?:om|misch)\b|\bagriculture\b|\bagrarisch\b|\bbouw\b|stafadviseur|onderhandelaar|consulent|clusterleider|groepsleider|vestigingsleider|coördinatrice|r&d|informatievoorziening|automatisering|\bexpert\b|\binterim\b",
    ),
    (
        "bestuur_toezicht_onvoldoende_sectorcontext",
        r"bestuurder|bestuurslid|\bbestuur\b|voorzitterr?|vice[- ]?voorzitter|commissaris|"
        r"\blid\b|adviesraad|adviesforum|auditcommissie|klankbordgroep|werkgroep|kernteam|jurylid|"
        r"panellid|presidium|portefeuillehouder|toezichtraad|commissielid|raad van advies|president|bestuurlijk",
    ),
    ("overig_onvoldoende_sectorcontext", r".+"), #telt als buiten politiek werk
]

STAF_IN_FUNCTIE = r"medewerk|adviseur|assistent|secretaris|voorlichter|woordvoerder|\bbeleid|speechschrijver|stagiair|campagne"
ECHT_KAMERLID   = r"\bkamerlid\b|\blid\b.*\bkamer\b|senaatslid"

def sector(tekst):
    if not tekst.strip(" |"):
        return "onbekend"
    functie = tekst.split(" | ")[0]
    for naam, patroon in REGELS_SECTOR:
        if re.search(patroon, tekst):
            if naam == "politiek_ondersteuning" and not re.search(POLITIEK_CONTEXT, tekst):
                continue
            if naam == "kamerlidmaatschap" and re.search(STAF_IN_FUNCTIE, functie) \
               and not re.search(ECHT_KAMERLID, functie):
                continue          # staf van een fractie is geen Kamerlid
            return naam
    return "niet_ingedeeld"

uniek_l = loopbaan[["tekst"]].drop_duplicates().copy()
uniek_l["sector"] = uniek_l["tekst"].map(sector)
loopbaan["sector"] = loopbaan["tekst"].map(sector)

print(uniek_l["sector"].value_counts())
uniek_l[uniek_l["sector"] == "niet_ingedeeld"].head(60)     # hier verbeter je je regels


import re
BINNEN_POLITIEK = {"kamerlidmaatschap", "politiek_ondersteuning", "politiek_bestuur"}
ONDUIDELIJK = {"overig_onvoldoende_sectorcontext", "bestuur_toezicht_onvoldoende_sectorcontext"}
loopbaan["buiten_politiek"] = ~loopbaan["sector"].isin(BINNEN_POLITIEK | {"onbekend", "niet_ingedeeld"})
loopbaan["buiten_zeker"]    = ~loopbaan["sector"].isin(BINNEN_POLITIEK | ONDUIDELIJK | {"onbekend", "niet_ingedeeld"})

lb = loopbaan.groupby("persoon_id").agg(
    n_loopbaanrijen=("sector", "size"),
    n_buiten_politiek=("buiten_politiek", "sum"),
    n_buiten_zeker=("buiten_zeker", "sum"),
)

kol = ["n_loopbaanrijen", "n_buiten_politiek", "n_buiten_zeker"]
persoon = persoon.drop(columns=[c for c in kol if c in persoon.columns])
n_voor = len(persoon)
persoon = persoon.merge(lb, left_on="id", right_index=True, how="left")
assert len(persoon) == n_voor, "merge heeft personen verdubbeld"
persoon[kol] = persoon[kol].fillna(0).astype(int)

m = persoon[persoon["n_loopbaanrijen"] > 0]
print("personen met loopbaan:", len(m))
print("ondergrens (zeker buiten politiek):", (m["n_buiten_zeker"] > 0).sum(), f"= {100*(m['n_buiten_zeker'] > 0).mean():.1f}%")
print("bovengrens (incl. onduidelijk):    ", (m["n_buiten_politiek"] > 0).sum(), f"= {100*(m['n_buiten_politiek'] > 0).mean():.1f}%")
print(persoon.groupby("functie")["n_loopbaanrijen"].apply(lambda s: round(100 * (s == 0).mean(), 1)))
print("geen loopbaanrijen:", round(100 * (persoon["n_loopbaanrijen"] == 0).mean(), 1), "%")
print("geen loopbaan buiten politiek:", round(100 * (persoon["n_buiten_politiek"] == 0).mean(), 1), "%")
print(pd.crosstab(persoon["heeft_fractie"], persoon["n_loopbaanrijen"] > 0, margins=True).to_string())

f = persoon[persoon["heeft_fractie"] & (persoon["geboortejaar"] >= 1950)]
print(len(f))
print(f["opleiding_status"].value_counts().to_string())

print(pd.crosstab(persoon["heeft_fractie"], persoon["n_loopbaanrijen"] > 0, margins=True).to_string())




De 1854 unieke loopbaanteksten zijn ingedeeld in sectoren. 278 personen hebben loopbaanrijen. Van hen heeft 94,6% zeker (263) en 96,8% hooguit (269) minstens één baan buiten de politiek. Zonder loopbaanrij: 62,0% van de huidige Tweede Kamer, 96,5% van de Eerste Kamer (kleine sub populatie dus snel hoog percentage bij missend), 92,3% van de oud-Kamerleden. Onder de 736 personen met fractie geboren vanaf 1950 heeft 262 een bepaald opleidingsniveau.

In [ ]:
# zijn het dezelfde 278 personen?
print(pd.crosstab(persoon["n_opleidingen"] > 0, persoon["n_loopbaanrijen"] > 0, margins=True).to_string())


# onder de 278 met loopbaan: hoeveel hebben iets buiten de politiek?
m = persoon[persoon["n_loopbaanrijen"] > 0]
print(len(m), round(100 * (m["n_buiten_politiek"] > 0).mean(), 1), "% heeft minstens één baan buiten politiek")

Opleiding en loopbaan zijn vrijwel dezelfde personen: 275 hebben beide, 3 alleen een loopbaan, 3 alleen een opleiding.

In [ ]:
persoon["loopbaan_status"] = "geen loopbaan geregistreerd"
persoon.loc[persoon["n_loopbaanrijen"] > 0, "loopbaan_status"] = "geregistreerd, geen baan buiten politiek ingedeeld"
persoon.loc[persoon["n_buiten_politiek"] > 0, "loopbaan_status"] = "geregistreerd, baan buiten politiek"
print(persoon["loopbaan_status"].value_counts().to_string())

2805 personen hebben geen loopbaan, 269 hebben een baan buiten de politiek (bovengrens: bedrijfsleven + vage sectoren) en 9 hebben alleen politieke rijen.

In [ ]:
# zijn de 791 'rijke' records dezelfde personen?
# er word gekeken of er een link ligt tussen portret url en het hebben van een fractie
print(pd.crosstab(persoon["portret_url"].notna(), persoon["heeft_fractie"]))

# wat zijn die 1166 placeholders, en wat is 'land'?
print(persoon["geboorteplaats"].value_counts().head(10))
print(persoon["land"].value_counts())

# hoeveel personen hebben zowel geboorteplaats als woonplaats echt ingevuld, en hoe zit het bij de huidige 150?
p = naar_na(persoon)
beide = p["geboorteplaats"].notna() & p["woonplaats"].notna()
print(beide.sum(), round(100 * beide.mean(), 1), "%")
print(p.groupby("functie")[["geboorteplaats", "woonplaats", "geboorteland"]]
        .agg(lambda s: round(100 * s.notna().mean(), 1)))

De laag met veel data (loopbaan en onderwijs) wordt gedefinieerd door fractie niet door portret. De crosstab laat zien dat 789 personen beide hebben, 14 een fractie zonder portret en 27 een portret zonder fractie.

### Steekproef

Met de steekproef is er gekeken naar de kwaliteit van de indelingscode en met kritiek past Claude of ChatGPT de REGEX code aan. Na 3 rondes is dit gestopt wegens een niet significante toename.

In [ ]:
# onderwijs
s1 = onderwijs.drop_duplicates(["opleiding_nl", "instelling"]).sample(50, random_state=2)
s1[["opleiding_nl", "instelling", "categorie", "bron"]].assign(juist="").to_csv("check_onderwijs.csv", index=False)

# loopbaan
s2 = loopbaan.drop_duplicates("tekst").sample(50, random_state=2)
s2[["tekst", "sector"]].assign(juist="").to_csv("check_loopbaan.csv", index=False)

s1

### Demografie

In [ ]:
REFDATUM = pd.Timestamp("2026-10-08")      # vaste datum, zodat de uitkomst reproduceerbaar is
persoon["geboorte_d"] = pd.to_datetime(persoon["geboortedatum"], errors="coerce")
persoon["overlijden_d"] = pd.to_datetime(persoon["overlijdensdatum"], errors="coerce")
persoon["geboortejaar"] = persoon["geboorte_d"].dt.year

# 1. Hoe volledig is de overlijdensdatum? (geboren <= 1900 is zeker overleden)
dec = persoon["geboortejaar"] // 10 * 10
print(persoon.groupby(dec)["overlijden_d"].apply(lambda s: round(s.notna().mean(), 2)).to_string())

# 2. Foutcontrole
sterfleeftijd = (persoon["overlijden_d"] - persoon["geboorte_d"]).dt.days / 365.25
print("overlijden voor geboorte:", (sterfleeftijd < 0).sum(), "| >110 jaar:", (sterfleeftijd > 110).sum(),
      "| overlijden na refdatum:", (persoon["overlijden_d"] > REFDATUM).sum())

# 3. De 150 huidige Tweede Kamerleden leven per definitie: hoeveel hebben toch een overlijdensdatum?
print(persoon.loc[persoon["functie"] == "Tweede Kamerlid", "overlijden_d"].notna().sum())


def levensstatus(df, drempel):
    s = pd.Series("onbekend (oud, geen overlijdensdatum)", index=df.index)
    s[df["overlijden_d"].notna()] = "overleden (datum bekend)"
    s[df["overlijden_d"].isna() & (df["geboortejaar"] >= drempel)] = "aangenomen in leven"
    s[df["overlijden_d"].isna() & df["geboortejaar"].isna()] = "onbekend (geen geboortedatum)"
    return s

for d in (1915, 1920, 1925, 1930):
    print(d, levensstatus(persoon, d).value_counts().to_dict())

De overlijdensdatum is volledig voor geboren tot en met 1919 (100%) en loopt daarna terug: 93% (1920), 54% (1930), 23% (1940), 3% (1950). Er zijn geen onmogelijke data en geen van de 150 huidige Kamerleden heeft er een. Met drempel 1930: 1820 overleden, 1250 aangenomen in leven, 11 oud zonder datum en 2 zonder geboortedatum.

In [ ]:
# steekproef voor Parlement.com
kol = [c for c in ["id", "achternaam", "roepnaam", "geboortedatum", "functie"] if c in persoon.columns]
check = persoon[persoon["overlijden_d"].isna() & persoon["geboortejaar"].between(1930, 1949)]
print(len(check))
print(check.sample(10, random_state=1)[kol].to_string())

# leeftijd bij aantreden (803 met fractie); alleen jaarverschil, dus ±1 jaar
persoon["leeftijd_aantreden"] = persoon["start_jaar_fractie"] - persoon["geboortejaar"]
a = persoon["leeftijd_aantreden"].dropna()
print(len(a)); print(a.describe().round(1).to_string())
print("onwaarschijnlijk (<18 of >80):", ((a < 18) | (a > 80)).sum())

# huidige leeftijd, alleen de 150 huidige Tweede Kamerleden
tk = persoon[persoon["functie"] == "Tweede Kamerlid"].copy()
tk["leeftijd_nu"] = ((REFDATUM - tk["geboorte_d"]).dt.days / 365.25).round(1)
print(tk["leeftijd_nu"].describe().round(1).to_string())

- Het gaat om 321 personen geboren 1930–1949 zonder overlijdensdatum. Leeftijd bij aantreden is bekend voor 801 personen (mediaan 42, interkwartielbereik 36–50). De huidige leeftijd is bekend voor alle 150 kamerleden (mediaan 42,5; bereik 26,0–74,1).
- De tien namen zijn gecontroleerd op parlement.com . Er is geen overlijdensdatum opgegeven voor deze personen waardoor de aanname "geen overlijdensdatum= nog in leven". Wél wordt dit alleen aangenomen voor personen na 1930. Iemand uit 1930 is momenteel 96 jaar oud. Deze aanname klopt voor de bovenstaande steekproef van 10 personen. 
- Verder is de verdeling van de leeftijden rechtscheef. Het gemiddelde is groter dan de mediaan.

In [ ]:
import unicodedata
def plaats_sleutel(x):
    if leeg(x):
        return None
    s = re.sub(r"\(.*?\)", " ", str(x).lower())          # "Ommen (Ov.)" -> "ommen"
    s = unicodedata.normalize("NFKD", s).encode("ascii", "ignore").decode()
    s = re.sub(r"[^a-z0-9]+", " ", s).strip()
    return s or None

persoon["geboorteplaats_sleutel"] = persoon["geboorteplaats"].map(plaats_sleutel)
print(persoon["geboorteplaats_sleutel"].notna().sum(), "ingevuld;", persoon["geboorteplaats_sleutel"].nunique(), "unieke plaatsen")
print(persoon["geboorteplaats_sleutel"].value_counts().head(25).to_string())

In [ ]:
#Omdat Den Haag ook wel 's gravenhage wordt genoemd of 's hertogenbosch Den Bosch. worden de aliasen samengevoegd.
ALIAS = {"den haag": "s gravenhage", "den bosch": "s hertogenbosch",
         "hertogenbosch": "s hertogenbosch", "gravenhage": "s gravenhage",}
persoon["geboorteplaats_sleutel"] = persoon["geboorteplaats_sleutel"].replace(ALIAS)

v = persoon["geboorteplaats_sleutel"].dropna()
for w in ["bosch", "haag", "bergen", "rijswijk", "beek"]:
    print(w, v[v.str.contains(w)].value_counts().to_dict())

vc = v.value_counts()
print(vc.head(10).to_string())
print((vc == 1).sum(), "plaatsen komen maar één keer voor")
print(vc[vc == 1].sample(40, random_state=1).index.tolist())

Den Haag gestegen in gemeentes: de alias dictionary heeft zin

In [ ]:
#Lookup tabel gebruiken van CBS om woonplaatsen op gemeente en provincies in te delen.
import cbsodata
lk = pd.DataFrame(cbsodata.get_data("86312NED"))
print(lk.columns.tolist())
print(lk.head())
lk

In [ ]:
lk = lk.rename(columns={"Woonplaatsen": "woonplaats", "Naam_2": "gemeente",
                        "Code_3": "gemeentecode", "Naam_4": "provincie"})
lk = lk[["woonplaats", "gemeente", "gemeentecode", "provincie"]].copy()
for c in lk.columns:
    lk[c] = lk[c].astype(str).str.strip()

# gemeentenamen ook als zoeksleutel (voor gevallen als "zaanstad")
gem = lk[["gemeente", "gemeentecode", "provincie"]].drop_duplicates().assign(woonplaats=lambda d: d["gemeente"])
lk_all = pd.concat([lk, gem[lk.columns]]).drop_duplicates()

ALIAS2 = {"s heerenbergh": "s heerenberg", "driebergen": "driebergen rijsenburg"}
persoon["geboorteplaats_sleutel"] = persoon["geboorteplaats_sleutel"].replace(ALIAS2)
ALIAS3 = {"capelle a d ijssel": "capelle aan den ijssel", "alphen a d rijn": "alphen aan den rijn",
          "willemstad n b": "willemstad", "cuijk en sint agatha": "cuijk", "cuijk en sint agata": "cuijk",
          "roosendaal en nispen": "roosendaal", "ginneken en bavel": "breda",
          "zuilen": "utrecht", "hillegersberg": "rotterdam", "jutphaas": "nieuwegein",
          "hoogezand sappemeer": "hoogezand", "beemster": "middenbeemster", "dantumadeel": "dantumadiel","mijmegen": "nijmegen", "wonseradeel": "sudwest fryslan", "wymbritseradeel": "sudwest fryslan",
          "dongeradeel": "noardeast fryslan", "westdongeradeel": "noardeast fryslan",
          "doniawerstal": "de fryske marren"}
persoon["geboorteplaats_sleutel"] = persoon["geboorteplaats_sleutel"].replace(ALIAS3)
lk_all["sleutel"] = lk_all["woonplaats"].map(plaats_sleutel)
n_gem = lk_all.groupby("sleutel")["gemeentecode"].nunique()
eenduidig = (lk_all[lk_all["sleutel"].isin(n_gem[n_gem == 1].index)]
             .drop_duplicates("sleutel")[["sleutel", "gemeente", "gemeentecode", "provincie"]])
ambigu = set(n_gem[n_gem > 1].index)

p = persoon[["id", "geboorteplaats_sleutel"]].merge(
        eenduidig, left_on="geboorteplaats_sleutel", right_on="sleutel", how="left")
assert len(p) == len(persoon)

def status(r):
    if pd.isna(r["geboorteplaats_sleutel"]): return "geen plaats ingevuld"
    if pd.notna(r["gemeente"]):               return "gematcht (eenduidig)"
    if r["geboorteplaats_sleutel"] in ambigu: return "niet eenduidig (meerdere gemeenten)"
    return "niet gevonden"

p["plaats_status"] = p.apply(status, axis=1)
print(p["plaats_status"].value_counts().to_string())
print("niet eenduidig:", p.loc[p["plaats_status"] == "niet eenduidig (meerdere gemeenten)", "geboorteplaats_sleutel"].value_counts().head(15).to_dict())
print("niet gevonden:", p.loc[p["plaats_status"] == "niet gevonden", "geboorteplaats_sleutel"].value_counts().head(40).to_dict())

Van de 1888 ingevulde geboorteplaatsen zijn er 1657 eenduidig aan een gemeente gekoppeld (87,8%), 183 niet gevonden (grotendeels buitenland en oude gemeenten) en 48 niet eenduidig.\
Oosterhout is een plaats in zowel Brabant als Gelderland en dus op basis van de huidige data niet in te delen

In [ ]:
ALIAS3 = {"capelle a d ijssel": "capelle aan den ijssel", "alphen a d rijn": "alphen aan den rijn",
          "willemstad n b": "willemstad", "cuijk en sint agatha": "cuijk", "cuijk en sint agata": "cuijk",
          "roosendaal en nispen": "roosendaal", "ginneken en bavel": "breda",
          "zuilen": "utrecht", "hillegersberg": "rotterdam", "jutphaas": "nieuwegein",
          "hoogezand sappemeer": "hoogezand", "beemster": "middenbeemster", "dantumadeel": "dantumadiel"}
missing = [v for v in ALIAS3.values() if plaats_sleutel(v) not in set(lk_all["sleutel"])]
print("alias-doelen die niet in de lookup staan:", missing)

In [ ]:
n_voor = len(persoon)
persoon = persoon.drop(columns=[c for c in ["geboorte_gemeente", "geboorte_gemeentecode",
                                            "geboorte_provincie", "plaats_status"] if c in persoon.columns])
persoon = persoon.merge(
    p.rename(columns={"gemeente": "geboorte_gemeente", "gemeentecode": "geboorte_gemeentecode",
                      "provincie": "geboorte_provincie"})
     [["id", "geboorte_gemeente", "geboorte_gemeentecode", "geboorte_provincie", "plaats_status"]],
    on="id", how="left")
assert len(persoon) == n_voor

g = persoon[persoon["plaats_status"].isin(["gematcht (eenduidig)", "gematcht (prefix)"])]
print(len(g))
print(g["geboorte_provincie"].value_counts().to_string())
vc = g["geboorte_gemeente"].value_counts()
print(vc.head(12).to_string())
print({f"gemeenten met n>={n}": int((vc >= n).sum()) for n in (1, 5, 10, 20)})

Van de 1657 gekoppelden komt 55,0% uit Zuid-Holland, Noord-Holland of Noord-Brabant. 253 van de 342 gemeenten hebben minstens één persoon, 88 hebben er minstens 5. De drie noemers voor het dashboard zijn 3083, 1888 en 1657.

In [ ]:
raw = pd.DataFrame(cbsodata.get_data("86312NED"))
prov_codes = (raw[["Naam_4", "Code_5"]].apply(lambda s: s.str.strip()).drop_duplicates()
              .rename(columns={"Naam_4": "geboorte_provincie", "Code_5": "provinciecode"}))

g = persoon[persoon["plaats_status"] == "gematcht (eenduidig)"]
prov = g.groupby("geboorte_provincie").size().rename("n").reset_index().merge(prov_codes, on="geboorte_provincie", how="left")
prov["aandeel_%"] = (100 * prov["n"] / prov["n"].sum()).round(1)
assert len(prov) == 12 and prov["provinciecode"].notna().all()

gem = g.groupby(["geboorte_gemeentecode", "geboorte_gemeente"]).size().rename("n").reset_index()
print({"personen_totaal": len(persoon),
       "plaats_ingevuld": int(persoon["geboorteplaats_sleutel"].notna().sum()),
       "gematcht": len(g)})

In [ ]:
ALIAS_ALLE = {**ALIAS, **ALIAS2, **ALIAS3}
def koppel_plaats(serie):
    sl = serie.map(plaats_sleutel).replace(ALIAS_ALLE)
    d = pd.DataFrame({"sleutel": sl.values}).merge(eenduidig, on="sleutel", how="left")
    d.index = serie.index
    d["status"] = "niet gevonden"
    d.loc[d["sleutel"].isna(), "status"] = "niet ingevuld"
    d.loc[d["sleutel"].isin(ambigu), "status"] = "niet eenduidig"
    d.loc[d["gemeente"].notna(), "status"] = "gematcht"
    return d

w = koppel_plaats(persoon["woonplaats"])
persoon["woon_gemeente"], persoon["woon_gemeentecode"] = w["gemeente"], w["gemeentecode"]
persoon["woon_provincie"], persoon["woon_status"] = w["provincie"], w["status"]
print(persoon["woon_status"].value_counts().to_string())

b = persoon[persoon["geboorte_provincie"].notna() & persoon["woon_provincie"].notna()]
print(len(b), "personen met geboorte- én woonprovincie")
print(pd.crosstab(b["functie"], b["geboorte_provincie"] == b["woon_provincie"], margins=True).to_string())

In [ ]:
p0 = naar_na(persoon)
var = {
    "geslacht ingevuld":               p0["geslacht"].notna(),
    "geboortedatum bekend":            persoon["geboorte_d"].notna(),
    "geboorteplaats ingevuld":         persoon["geboorteplaats_sleutel"].notna(),
    "geboorteplaats aan gemeente":     persoon["plaats_status"].eq("gematcht (eenduidig)"),
    "woonplaats ingevuld":             p0["woonplaats"].notna(),
    "geboorteland ingevuld":           p0["geboorteland"].notna(),
    "fractiekoppeling":                persoon["heeft_fractie"],
    "leeftijd bij aantreden":          persoon["leeftijd_aantreden"].notna(),
    "opleidingsrij":                   persoon["n_opleidingen"] > 0,
    "opleidingsniveau bepaald":        persoon["opleiding_status"].eq("niveau bepaald"),
    "loopbaanrij":                     persoon["n_loopbaanrijen"] > 0,
    "loopbaan: zeker buiten politiek": persoon["n_buiten_zeker"] > 0,
}
groepen = {"Alle": pd.Series(True, index=persoon.index),
           "Tweede Kamer": persoon["functie"].eq("Tweede Kamerlid"),
           "Eerste Kamer": persoon["functie"].eq("Eerste Kamerlid"),
           "Oud-Kamerlid": persoon["functie"].eq("Oud Kamerlid")}
rijen = []
for naam, m in var.items():
    r = {"variabele": naam}
    for g, sel in groepen.items():
        n, N = int((m & sel).sum()), int(sel.sum())
        r[f"{g} (n={N})"] = f"{n} ({100 * n / N:.1f}%)"
    rijen.append(r)
overzicht = pd.DataFrame(rijen).set_index("variabele")
print(overzicht.to_string())

In [ ]:
#geslacht
t = pd.crosstab(persoon["functie"], persoon["geslacht"], margins=True)
t["% vrouw (van totaal)"] = (100 * t["vrouw"] / t["All"]).round(1)
print(t.to_string())

Geslacht is voor alle 3083 personen ingevuld: man, vrouw en één persoon met geslacht x (non-binair), die Tweede Kamerlid is. Voor die categorie tonen we alleen het aantal. Percentages en gemiddelden binnen een categorie van één persoon zijn statistisch niet zinvol en worden niet berekend. Het percentage vrouw is het aandeel van het functietotaal (150 / 85 / 2848).

## Streamlit
### Export voor het dashboard

In deze stap worden de uitkomsten van het notebook opgeslagen als parquet-bestanden in de map `data/`. Het Streamlit-dashboard leest alleen deze bestanden en voert de analyse niet opnieuw uit.

**1. Afgeleide kolommen.** Aan `persoon` zijn drie kolommen toegevoegd: `is_eerste_kamerlid`, `is_huidig_tweede_kamerlid` en `leeftijd_nu`. `leeftijd_nu` is berekend op de vaste referentiedatum 2026-10-08 en bestaat alleen voor de 150 huidige Tweede Kamerleden. Voor oud-Kamerleden is de levensstatus niet betrouwbaar bekend.

**2. Controles vooraf.** Voor het schrijven is gecontroleerd of de belangrijkste aantallen nog kloppen: 3083 personen (150 Tweede Kamer, 85 Eerste Kamer, 2848 oud-Kamerleden), 3081 bekende geboortedatums, 803 met fractiekoppeling, 801 met leeftijd bij aantreden, 265 met bepaald opleidingsniveau, 278 met loopbaanrijen, 1657 geboorteplaatsen en 1601 woonplaatsen gekoppeld aan een gemeente, 979 opleidingsrijen en 1987 loopbaanrijen. Bij een afwijking stopt de code. Alle controles zijn geslaagd.

**3. Dekkingstabel.** Per variabele en groep (Alle, Tweede Kamer, Eerste Kamer, Oud-Kamerlid) is het aantal ingevulde waarden (n), de groepsgrootte (N) en het percentage opgeslagen. Overlijdensdatum is toegevoegd: 1820 van 3083 (59,0%). Voor huidige Kamerleden is 0% hier geen datafout, want zij leven nog.

**4. Sectoroverzicht.** Per sector is het aantal unieke personen en het aantal loopbaanrijen geteld, voor alle groepen. Een persoon kan meerdere sectoren hebben, dus de percentages tellen niet op tot 100%. De noemer is het aantal personen met minstens één loopbaanrij: 278 totaal (Tweede Kamer 57, Eerste Kamer 3, oud-Kamerlid 218). Groepen met minder dan 10 personen zijn gemarkeerd (`klein_n`). De sector `onbekend` (17 personen) is geen echte sector.

**5. Geografie.** Per provincie en per gemeente is het aantal personen geteld, apart voor geboorteplaats en woonplaats en per groep. Alleen eenduidig gekoppelde plaatsen tellen mee (geboorte 1657, woon 1601). Gemeenten zonder personen komen niet in de tabel voor en moeten bij een kaart worden aangevuld met n = 0.

**6. Opslaan.** Zeven tabellen zijn weggeschreven en teruggelezen, waarbij het aantal rijen en kolommen is vergeleken:

| Bestand | Rijen | Kolommen | Eenheid |
|---|---|---|---|
| `personen` | 3083 | 63 | persoon |
| `loopbaan` | 1987 | 20 | loopbaanrij |
| `onderwijs` | 979 | 21 | opleidingsrij |
| `dekking` | 52 | 5 | variabele × groep |
| `sector_overzicht` | 51 | 9 | sector × groep |
| `geo_provincie` | 91 | 8 | provincie × soort × groep |
| `geo_gemeente` | 1226 | 8 | gemeente × soort × groep |

Ruwe API-kolommen (zoals `portret_url` en de wijzigingsdata) zijn uit `personen` weggelaten.

**7. Datawoordenboek.** `datawoordenboek.csv` bevat 104 kolommen (alle kolommen van de drie hoofdtabellen) met type, eenheid (persoon of rij) en aantal missende waarden, waarbij placeholders zoals "-" als missend zijn geteld. De kolom `omschrijving` is nog leeg en wordt handmatig ingevuld.

**Let op bij gebruik:** `leeftijd_nu` geldt alleen voor de huidige Tweede Kamer. Dekking van overlijdensdatum zegt niets over datakwaliteit bij levende personen. Percentages in het sectoroverzicht tellen niet op tot 100%.

In [ ]:
from pathlib import Path
 
OUT = Path("data")
OUT.mkdir(exist_ok=True)
 
persoon["is_eerste_kamerlid"] = persoon["functie"].eq("Eerste Kamerlid")
persoon["is_huidig_tweede_kamerlid"] = persoon["functie"].eq("Tweede Kamerlid")
# leeftijd_nu alleen voor de 150 huidige Tweede Kamerleden (van oud-Kamerleden is
# de levensstatus niet betrouwbaar bekend, zie de levensstatus-analyse)
persoon["leeftijd_nu"] = (
    ((REFDATUM - persoon["geboorte_d"]).dt.days / 365.25).round(1)
    .where(persoon["is_huidig_tweede_kamerlid"])
)
 
# 2. Controles vooraf (getallen uit je eigen celuitvoer)
assert len(persoon) == 3083 and persoon["id"].is_unique
assert persoon["functie"].value_counts().to_dict() == {
    "Oud Kamerlid": 2848, "Tweede Kamerlid": 150, "Eerste Kamerlid": 85}
assert persoon["geboorte_d"].notna().sum() == 3081
assert persoon["heeft_fractie"].sum() == 803
assert persoon["leeftijd_aantreden"].notna().sum() == 801
assert (persoon["opleiding_status"] == "niveau bepaald").sum() == 265
assert (persoon["n_loopbaanrijen"] > 0).sum() == 278
assert (persoon["plaats_status"] == "gematcht (eenduidig)").sum() == 1657
assert (persoon["woon_status"] == "gematcht").sum() == 1601
assert persoon["leeftijd_nu"].notna().sum() == 150
assert len(onderwijs) == 979 and len(loopbaan) == 1987
print("controles vooraf: ok")
 
#  3. Dekking als numerieke tabel (voor banner en pagina Datakwaliteit)
p0 = naar_na(persoon)
var2 = dict(var)  # uit cel 109
# let op: voor huidige Kamerleden is een ontbrekende overlijdensdatum juist correct;
# deze dekking zegt dus niets over kwaliteit, zie levensstatus-analyse (drempel 1930)
var2["overlijdensdatum ingevuld"] = p0["overlijdensdatum"].notna()
 
rijen = []
for naam, m in var2.items():
    for g, sel in groepen.items():
        n, N = int((m & sel).sum()), int(sel.sum())
        rijen.append({"variabele": naam, "groep": g, "n": n, "N": N,
                      "pct": round(100 * n / N, 1)})
dekking = pd.DataFrame(rijen)
assert dekking.query("variabele == 'fractiekoppeling' and groep == 'Alle'")["n"].item() == 803
print(dekking.query("variabele == 'overlijdensdatum ingevuld'").to_string(index=False))
 
#  4. Sectoren: unieke personen per sector, per groep
FUNCTIE_VAN_GROEP = {"Tweede Kamer": "Tweede Kamerlid",
                     "Eerste Kamer": "Eerste Kamerlid",
                     "Oud-Kamerlid": "Oud Kamerlid"}
functie_map = persoon.set_index("id")["functie"]
lb = loopbaan.assign(persoon_functie=loopbaan["persoon_id"].map(functie_map))
 
delen = []
for g in ["Alle", *FUNCTIE_VAN_GROEP]:
    d = lb if g == "Alle" else lb[lb["persoon_functie"] == FUNCTIE_VAN_GROEP[g]]
    t = (d.groupby("sector")
          .agg(n_personen=("persoon_id", "nunique"), n_rijen=("sector", "size"))
          .reset_index())
    t["groep"] = g
    t["N_personen_met_loopbaan"] = d["persoon_id"].nunique()
    delen.append(t)
sector_overzicht = pd.concat(delen, ignore_index=True)
sector_overzicht["pct_personen"] = (
    100 * sector_overzicht["n_personen"] / sector_overzicht["N_personen_met_loopbaan"]).round(1)
sector_overzicht["binnen_politiek"] = sector_overzicht["sector"].isin(BINNEN_POLITIEK)
sector_overzicht["onduidelijk"] = sector_overzicht["sector"].isin(ONDUIDELIJK)
sector_overzicht["klein_n"] = sector_overzicht["N_personen_met_loopbaan"] < 10
 
noemers = sector_overzicht.groupby("groep")["N_personen_met_loopbaan"].first().to_dict()
assert noemers == {"Alle": 278, "Tweede Kamer": 57, "Eerste Kamer": 3, "Oud-Kamerlid": 218}, noemers
# een persoon kan in meerdere sectoren voorkomen: pct_personen telt dus NIET op tot 100
print(sector_overzicht.query("groep == 'Alle'")
      .sort_values("n_personen", ascending=False)
      [["sector", "n_personen", "n_rijen", "pct_personen"]].to_string(index=False))
 
# %% 5. Geografie: geboorte- en woonplaats naar provincie en gemeente
SOORTEN = {
    "geboorte": dict(gc="geboorte_gemeentecode", gn="geboorte_gemeente", pn="geboorte_provincie",
                     ok=persoon["plaats_status"].eq("gematcht (eenduidig)"),
                     ingevuld=persoon["geboorteplaats_sleutel"].notna()),
    "woon": dict(gc="woon_gemeentecode", gn="woon_gemeente", pn="woon_provincie",
                 ok=persoon["woon_status"].eq("gematcht"),
                 ingevuld=p0["woonplaats"].notna()),
}
pc = prov_codes.set_index("geboorte_provincie")["provinciecode"]
 
geo_prov, geo_gem = [], []
for soort, c in SOORTEN.items():
    for g, sel in groepen.items():
        d = persoon[sel & c["ok"]]
        noemer = dict(soort=soort, groep=g, N_groep=int(sel.sum()),
                      N_ingevuld=int((sel & c["ingevuld"]).sum()), N_gematcht=len(d))
        a = (d.groupby(c["pn"]).size().rename("n").reset_index()
              .rename(columns={c["pn"]: "provincie"}))
        a["provinciecode"] = a["provincie"].map(pc)
        geo_prov.append(a.assign(**noemer))
        b = (d.groupby([c["gc"], c["gn"]]).size().rename("n").reset_index()
              .rename(columns={c["gc"]: "gemeentecode", c["gn"]: "gemeente"}))
        geo_gem.append(b.assign(**noemer))
 
geo_provincie = pd.concat(geo_prov, ignore_index=True)
geo_gemeente = pd.concat(geo_gem, ignore_index=True)
assert geo_provincie["provinciecode"].notna().all()
assert geo_provincie.query("soort == 'geboorte' and groep == 'Alle'")["n"].sum() == 1657
assert geo_provincie.query("soort == 'woon' and groep == 'Alle'")["n"].sum() == 1601
# let op: gemeenten zonder personen ontbreken hier (253 van 342 hebben er minstens 1);
# vul bij de kaart-join de ontbrekende gemeenten aan met n = 0
 
# %% 6. Schrijven naar parquet (+ terugleescontrole)
import pyarrow as pa
 
def veilig(df):
    """Object-kolommen met gemengde types naar string, anders weigert pyarrow."""
    d = df.copy()
    for c in d.select_dtypes("object").columns:
        try:
            pa.array(d[c])
        except (pa.ArrowInvalid, pa.ArrowTypeError):
            d[c] = d[c].astype("string")
    return d
 
RUW = ["portret_url", "content_type", "content_length", "gewijzigd_op", "api_gewijzigd_op"]
bestanden = {
    "personen": persoon.drop(columns=[c for c in RUW if c in persoon.columns]),
    "loopbaan": loopbaan,
    "onderwijs": onderwijs,
    "dekking": dekking,
    "sector_overzicht": sector_overzicht,
    "geo_provincie": geo_provincie,
    "geo_gemeente": geo_gemeente,
}
for naam, df in bestanden.items():
    veilig(df).to_parquet(OUT / f"{naam}.parquet", index=False)
    terug = pd.read_parquet(OUT / f"{naam}.parquet")
    assert terug.shape == df.shape, naam
    print(f"{naam:18s} {terug.shape[0]:>6} rijen {terug.shape[1]:>3} kolommen")
 
# %% 7. Datawoordenboek (skelet; kolom 'omschrijving' vul je zelf in)
EENHEID = {"personen": "persoon", "loopbaan": "rij (loopbaanregel)", "onderwijs": "rij (opleidingsregel)"}
wb = []
for naam, eenheid in EENHEID.items():
    df = pd.read_parquet(OUT / f"{naam}.parquet")
    d = naar_na(df)  # placeholders zoals "-" tellen als missend
    for c in df.columns:
        wb.append({"tabel": naam, "kolom": c, "dtype": str(df[c].dtype), "eenheid": eenheid,
                   "n_missend": int(d[c].isna().sum()), "omschrijving": ""})
pd.DataFrame(wb).to_csv(OUT / "datawoordenboek.csv", index=False, encoding="utf-8-sig")
print("datawoordenboek.csv:", len(wb), "kolommen")
 